# Milestone 1

This milestone focuses on understanding the dataset and establishing a baseline performance through **exploratory data analysis (EDA)** and simple **heuristic-based methods** using `librosa`.

---

## Suggested Readings
- [Hugging Face Audio Course](https://huggingface.co/learn/audio-course/en/chapter0/introduction)
- [Librosa Documentation](https://librosa.org/doc/main/core.html#audio-loading)

---

## Instructions
Use this notebook to answer **all Milestone-1 questions**.

---

## Resources
- Notebook Link:  
  https://colab.research.google.com/drive/1m6UczhxQIke_raWSqukSWuiKbIVt7MMb?usp=sharing  

- Competition Link:  
  https://www.kaggle.com/competitions/jan-2026-dl-gen-ai-project/

In [1]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch

import warnings
warnings.filterwarnings("ignore")

In [2]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [3]:
# CONFIGURATION
DATA_ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems'
GENRES = ['blues','classical','country','disco','hiphop','jazz','metal','pop','reggae','rock']
STEMS = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'
SONG_INDEX = 0

In [4]:
def build_dataset(root_dir, val_split=0.17, seed=42):
    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    rng = random.Random(seed)
    
    all_songs = []
    corrupted_count = 0
    small_5_0491_count = 0
    large_5_0493_count = 0
    
    for genre in GENRES:
        genre_path = os.path.join(root_dir, genre)
        if not os.path.exists(genre_path):
            continue
            
        song_folders = [f for f in os.listdir(genre_path) if os.path.isdir(os.path.join(genre_path, f))]
        song_folders.sort()
        
        for song_folder in song_folders:
            song_path = os.path.join(genre_path, song_folder)
            song_files = os.listdir(song_path)
            missing_stems = []
            
            for stem in STEMS:
                stem_path = os.path.join(song_path, stem)
                if stem not in song_files:
                    missing_stems.append(stem)
                else:
                    file_size = os.path.getsize(stem_path)
                    if file_size < 4096:
                        corrupted_count += 1
                    elif file_size < 5.0491 * 1024 * 1024:
                        small_5_0491_count += 1
                    elif file_size > 5.0493 * 1024 * 1024:
                        large_5_0493_count += 1
            
            if not missing_stems:
                song_data = {
                    'genre': genre,
                    'song_folder': song_folder,
                    'stems': {stem.replace('.wav', ''): os.path.join(song_path, stem) for stem in STEMS}
                }
                all_songs.append(song_data)
    
    for genre in GENRES:
        genre_songs = [s for s in all_songs if s['genre'] == genre]
        rng.shuffle(genre_songs)
        split_idx = int(len(genre_songs) * (1 - val_split))
        train_songs = genre_songs[:split_idx]
        val_songs = genre_songs[split_idx:]
        
        def add_to_dict(target_dict, song_list):
            for song in song_list:
                for stem_name, stem_path in song['stems'].items():
                    target_dict[song['genre']][stem_name].append(stem_path)
        
        add_to_dict(train_dataset, train_songs)
        add_to_dict(val_dataset, val_songs)
    
    print(f"Q1: {corrupted_count + small_5_0491_count}")
    print(f"Q2: {abs(large_5_0493_count - small_5_0491_count)}")
    train_reggae_drums = len(train_dataset['reggae']['drums'])
    val_country_vocals = len(val_dataset['country']['vocals'])
    print(f"Q3: {abs(train_reggae_drums - val_country_vocals)}")
    
    return train_dataset, val_dataset

tr, val = build_dataset(DATA_ROOT)

Q1: 1256
Q2: 1072
Q3: 66


In [5]:
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
    records = []
    
    for genre in dataset_dict:
        for stem_name in dataset_dict[genre]:
            for file_path in dataset_dict[genre][stem_name]:
                try:
                    audio, sr_loaded = librosa.load(file_path, sr=sr)
                    total_duration = len(audio) / sr
                    non_silent_intervals = librosa.effects.split(audio, top_db=top_db)
                    
                    silence_type = []
                    max_silence = 0
                    
                    if len(non_silent_intervals) == 0:
                        silence_type.append("fully_silent")
                        max_silence = total_duration
                    else:
                        if non_silent_intervals[0][0] > 0:
                            start_silence = non_silent_intervals[0][0] / sr
                            silence_type.append("start")
                            max_silence = max(max_silence, start_silence)
                        
                        if non_silent_intervals[-1][1] < len(audio):
                            end_silence = (len(audio) - non_silent_intervals[-1][1]) / sr
                            silence_type.append("end")
                            max_silence = max(max_silence, end_silence)
                        
                        for i in range(len(non_silent_intervals) - 1):
                            gap = (non_silent_intervals[i+1][0] - non_silent_intervals[i][1]) / sr
                            if gap > 0:
                                silence_type.append("middle")
                                max_silence = max(max_silence, gap)
                    
                    if max_silence >= threshold_sec:
                        records.append({
                            "Genre": genre,
                            "Stem": stem_name,
                            "Duration": round(total_duration, 2),
                            "Max_Silence_Sec": round(max_silence, 2),
                            "Silence_Location": ", ".join(silence_type),
                            "File_Path": file_path
                        })
                        
                except Exception:
                    continue
    
    df = pd.DataFrame(records)
    
    print(f"Q4: {len(df)}")
    vocal_silence = df[df['Stem'] == 'vocals']
    print(f"Q5: {len(vocal_silence)}")
    if len(vocal_silence) > 0:
        print(f"Q6: {vocal_silence['Max_Silence_Sec'].mean():.2f}")
    jazz_drums_silence = df[(df['Genre'] == 'jazz') & (df['Stem'] == 'drums')]
    print(f"Q7: {len(jazz_drums_silence)}")
    jazz_drums_middle = jazz_drums_silence[jazz_drums_silence['Silence_Location'].str.contains('middle', case=False, na=False)]
    print(f"Q8: {len(jazz_drums_middle)}")
    jazz_drums_long = jazz_drums_silence[jazz_drums_silence['Max_Silence_Sec'] >= 10]
    print(f"Q9: {len(jazz_drums_long)}")
    
    return df

df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)
pivot_table = df_silence.pivot_table(values='File_Path', index='Genre', columns='Stem', aggfunc='count', fill_value=0)
print(pivot_table)

Q4: 680
Q5: 304
Q6: 12.59
Q7: 24
Q8: 24
Q9: 7
Stem       bass  drums  other  vocals
Genre                                
blues        17     22      5      43
classical    68     57      5      69
country      16     16      2      16
disco         8      2      3      18
hiphop       20      3     22       6
jazz         25     24      1      70
metal         6      2      1      40
pop          10      5      2       4
reggae        5      4      7      12
rock         10      7      1      26


In [6]:
stems_audio = []
try:
    rock_drums_path = tr['rock']['drums'][0]
    rock_vocals_path = tr['rock']['vocals'][0]
    rock_bass_path = tr['rock']['bass'][0]
    rock_other_path = tr['rock']['other'][0]
    
    for key in STEM_KEYS:
        if key == 'drums':
            audio, sr_loaded = librosa.load(rock_drums_path, sr=SR, duration=DURATION)
        elif key == 'vocals':
            audio, sr_loaded = librosa.load(rock_vocals_path, sr=SR, duration=DURATION)
        elif key == 'bass':
            audio, sr_loaded = librosa.load(rock_bass_path, sr=SR, duration=DURATION)
        elif key == 'other':
            audio, sr_loaded = librosa.load(rock_other_path, sr=SR, duration=DURATION)
        stems_audio.append(audio)
    
    print(f"Q10: {len(stems_audio[0])}")
    print("Audio loaded successfully.")
except NameError:
    print("ERROR: 'tr' dictionary not found. Please run build_dataset() first.")
except IndexError:
    print(f"ERROR: Song index {SONG_INDEX} out of range for genre {GENRE_TO_TEST}.")
except Exception as e:
    print(f"ERROR: {e}")

Q10: 110250
Audio loaded successfully.


In [7]:
stems_stack = np.stack(stems_audio)
mix_raw = np.sum(stems_stack, axis=0)
rms_val = np.sqrt(np.mean(mix_raw ** 2))
max_val = np.max(np.abs(mix_raw))

if max_val > 0:
    mix_norm = mix_raw / max_val
else:
    mix_norm = mix_raw

assert np.isclose(np.max(np.abs(mix_norm)), 1.0), "Normalization failed."

print(f"Q11: {rms_val:.4f}")
print(f"Q12: {max_val:.4f}")

Q11: 0.1021
Q12: 0.5894
